Copyright 2026 Google LLC

In [ ]:
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

<a target="_blank" href="https://colab.research.google.com/github/google-gemma/gbench/blob/main/examples/notebooks/01_gbench_101_quickstart.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# gbench 101: Installation, Ollama setup, and quick performance check

**Author:** [Luciano Martins](https://github.com/lucianommartins)

This interactive notebook introduces `gbench`, an open-source performance benchmarking and capability evaluation suite for foundation models. You will learn how to install the package, configure a local Ollama serving engine with a quantized Google Gemma model, execute a baseline performance check, and inspect the resulting metrics.

## Learning objectives

1. Install `gbench` and its required dependencies in an isolated Python environment.
2. Configure and start a local Ollama server running a quantized Gemma 4 model (`unsloth/gemma-4-E4B-it-qat-GGUF`) with a context window of 8192 tokens.
3. Run a quick performance smoke test using `gbench --preset quick` over standard OpenAI `/v1` REST endpoints.
4. Load and inspect JSON benchmark outputs using Pandas to analyze Time to First Token (TTFT), Time per Output Token (TPOT), and request throughput.
5. Perform a clean session shutdown to terminate background servers and reclaim hardware memory.

## Useful resources

* [gbench GitHub repository](https://www.github.com/google-gemma/gbench)
* [Ollama documentation](https://github.com/ollama/ollama)
* [Unsloth Gemma 4 QAT GGUF checkpoints](https://huggingface.co/unsloth/gemma-4-E4B-it-qat-GGUF)

## 1. Environment setup and installation

We clone the `gbench` repository from GitHub, change directory into the project root (`%cd gbench`), and install the package in editable mode (`pip install -e .`). This builds and links the `gbench` CLI executable without installing unnecessary development linters.

In [ ]:
import os, sys
from pathlib import Path

# Safe environment setup: Always normalize to top-level repository
if Path("/content").exists():
    %cd -q /content
    if not Path("/content/gbench").is_dir():
        !git clone https://github.com/google-gemma/gbench.git
    %cd -q /content/gbench
else:
    if not Path("pyproject.toml").is_file() and not Path("gbench").is_dir():
        if not Path("gbench").is_dir():
            !git clone https://github.com/google-gemma/gbench.git
        %cd gbench

%pip install -e . -q
import gbench
print(f"gbench version {gbench.__version__} installed successfully.")

# Inspect available presets
!gbench --list presets

## Hugging Face authentication (required)

This notebook downloads the Gemma 4 GGUF (and its vision projector) plus tokenizers/datasets from the Hugging Face Hub with `huggingface_hub` — some are **gated** — so an **`HF_TOKEN` is required**.

1. Create a **read** token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) and accept the license on any gated model/dataset page you use.
2. On **Colab**: click the **🔑 key icon (Secrets)** in the left sidebar → **Add new secret**, name it `HF_TOKEN`, paste the token, and toggle **Notebook access** on.
3. **Elsewhere**: set it in your environment, e.g. `export HF_TOKEN=hf_...` (or `os.environ["HF_TOKEN"] = "hf_..."`).

The next cell loads the token and stops with instructions if it is missing.

In [ ]:
import os

# HF_TOKEN is REQUIRED: this notebook downloads the Gemma 4 GGUF (+ vision projector)
# and tokenizers/datasets from the Hugging Face Hub via huggingface_hub, which
# authenticates with it (resumable, higher rate limits, and access to gated repos).
try:
    from google.colab import userdata          # Colab: read from the Secrets vault
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass                                        # not on Colab, or the secret is unset
if not os.environ.get("HF_TOKEN"):
    raise RuntimeError(
        "HF_TOKEN is not set. On Colab: click the key icon (Secrets) in the left "
        "sidebar, add a secret named HF_TOKEN, and turn on notebook access. "
        "Elsewhere: os.environ['HF_TOKEN'] = 'hf_...'. "
        "Create a read token at https://huggingface.co/settings/tokens."
    )
print("HF_TOKEN loaded.")

## 2. Installing Ollama locally

We check if the Ollama binary is present on the system. If it is not found, we install Ollama using its official Linux installation script (`curl -fsSL https://ollama.com/install.sh | sh`). Finally, we run `ollama --version` to verify that the installation succeeded and the CLI is available.

In [ ]:
import subprocess, os, shutil, glob

# (Re)install Ollama unless BOTH the binary and its llama-server runner are present.
# A binary-only partial install fails every request with "llama-server binary not
# found", so checking only for the binary would skip the repair.
def _ollama_ready():
    if not shutil.which("ollama"):
        return False
    return any(glob.glob(p) for p in (
        "/usr/local/lib/ollama/llama-server",
        "/usr/local/lib/ollama/*/llama-server",
        "/usr/lib/ollama/llama-server",
    ))

if not _ollama_ready():
    print("Installing/repairing Ollama (binary + llama-server runner)...")
    # Ensure zstd is available (required by Ollama Linux tar.zst packages)
    subprocess.run("command -v zstd >/dev/null || (command -v apt-get >/dev/null && apt-get update -qq && apt-get install -y -qq zstd)", shell=True)
    # Run official Ollama installer
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True)
else:
    print("Ollama (with llama-server runner) already installed.")

# Ensure binary directory is present in PATH for subsequent cells
for p in ["/usr/local/bin", "/usr/bin", os.path.expanduser("~/.local/bin")]:
    if os.path.exists(os.path.join(p, "ollama")) and p not in os.environ.get("PATH", ""):
        os.environ["PATH"] = f"{p}:{os.environ.get('PATH', '')}"

!ollama --version

## 3. Launching background Ollama server

We launch the `ollama serve` process in the background and send a health check request to `http://localhost:11434/` to verify that the HTTP API is alive ("Ollama is running").

In [ ]:
import subprocess, time, requests
try:
    resp = requests.get("http://localhost:11434/", timeout=2)
    print("Ollama server already active:", resp.text.strip())
except Exception:
    print("Starting background ollama serve...")
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(4)
    resp = requests.get("http://localhost:11434/")
    print("Server health check:", resp.text.strip())

## 4. Downloading the GGUF and writing the Modelfile

We download the quantized Gemma 4 GGUF **and its vision projector (`mmproj`)** from the Hugging Face Hub with `huggingface_hub` (authenticated via `HF_TOKEN`, so the download is resumable and not rate-limited), then write an Ollama `Modelfile.qat` that points `FROM` the **local** files:
* **`num_ctx 8192`**: Context window of 8192 tokens.
* **`SYSTEM prompt`**: System instruction defining Gemma 4 AI assistant capabilities.

We download here rather than letting `ollama create` pull `hf.co/…` itself: Ollama's puller is anonymous (it can't use `HF_TOKEN`) and can stall on the HF CDN. The two-`FROM` import (main GGUF + `mmproj`) keeps the model's **vision** capability.

> **Note:** The `gemma-4` GGUF and tokenizer names used in this notebook are the intended Gemma 4 launch artifacts; they are placeholders until the model is published.

In [ ]:
import os
from huggingface_hub import HfApi, hf_hub_download

HF_REPO = "unsloth/gemma-4-E4B-it-qat-GGUF"
HF_QUANT = "UD-Q4_K_XL"
token = os.environ["HF_TOKEN"]  # required; loaded in the Hugging Face auth cell above

# Download the model GGUF and its vision projector (mmproj) via huggingface_hub, which
# authenticates with HF_TOKEN - Ollama's own hf.co puller is anonymous and can stall
# on the HF CDN. Build the model FROM the local files: a two-FROM import (main +
# mmproj) keeps gemma-4's vision capability (verified with `ollama show`). realpath
# resolves the HF cache symlink so `ollama create` reads the actual file.
files = HfApi().list_repo_files(HF_REPO, token=token)
main = [f for f in files if f.endswith(".gguf") and HF_QUANT in f]
proj = [f for f in files if f.endswith(".gguf") and "mmproj" in f.lower() and "-F16" in f]
if not main:
    raise RuntimeError(f"No {HF_QUANT} .gguf found in {HF_REPO}.")
GGUF_PATH = os.path.realpath(hf_hub_download(HF_REPO, main[0], token=token))
lines = [f"FROM {GGUF_PATH}"]
if proj:  # vision projector -> keeps multimodal capability
    lines.append(f"FROM {os.path.realpath(hf_hub_download(HF_REPO, proj[0], token=token))}")
lines += ['PARAMETER num_ctx 8192',
          'SYSTEM "You are a helpful Gemma 4 AI assistant with reasoning, vision, and tool calling capabilities."']
with open("Modelfile.qat", "w", encoding="utf-8") as f:
    f.write("\n".join(lines) + "\n")
print("Created Modelfile.qat from local GGUF" + (" + mmproj (vision)" if proj else ""))

## 5. Registering model and running generation smoke test

We register our custom model tag (`gemma4-qat:4b`) with `ollama create -f Modelfile.qat`. Because `Modelfile.qat` points `FROM` the local GGUF (and `mmproj`) downloaded in the previous cell, this reads from disk — no network pull. We then run a quick generation test to verify the model loads into hardware memory and generates tokens correctly.

In [ ]:
import subprocess, requests

MODEL_TAG = "gemma4-qat:4b"
print(f"Registering model {MODEL_TAG} from the local GGUF (built in the previous cell)...")
subprocess.run(["ollama", "create", MODEL_TAG, "-f", "Modelfile.qat"], check=True)

print("Running quick generation smoke test via Ollama API (cold load into GPU VRAM)...")
resp = requests.post(
    "http://localhost:11434/api/generate",
    json={"model": MODEL_TAG, "prompt": "Reply with the single word: READY.", "stream": False},
    timeout=300,
)
print("Smoke test response:", resp.json().get("response", "").strip())

## 6. Verifying OpenAI REST endpoint readiness

Before launching `gbench`, we query `http://localhost:11434/v1/models` to verify that Ollama is serving standard OpenAI `/v1` REST payloads and that our registered model is listed.

In [ ]:
import requests
resp = requests.get("http://localhost:11434/v1/models")
print("OpenAI /v1/models endpoint HTTP status:", resp.status_code)
models = [m["id"] for m in (resp.json().get("data") or [])]
print("Available REST models:", models)
if not models:
    print("No models registered yet - re-run the model registration cell above.")

## 7. Running a quick performance smoke test

With Ollama running locally, we execute a reduced-scenario performance run using `gbench --preset quick`. This preset runs 0 warmup iterations and 1 test iteration at concurrency `batch_size=1`.

Note that `--preset quick` with no `*-only` flag does **not** produce a serving-only result: it runs the full default pillar set. Over a `--remote-endpoint`, that means a serving latency sweep **and**, by default, an arrival-rate (QPS) **stress ramp**. To skip the stress ramp you would add `--no-stress-test` (stress is included by default in all presets). So this is a reduced-scenario run covering both serving latency and a QPS stress sweep, not a rapid serving-only check. See the next cell for how to tune the stress ramp to your hardware.

By passing `--remote-endpoint http://localhost:11434/v1`, we instruct `gbench` to talk directly to Ollama's OpenAI `/v1` REST interface without checking local GPU VRAM or spawning a vLLM subprocess. (Offline throughput is skipped for remote endpoints, since it requires a local vLLM engine.)

### Or serve with vLLM instead of Ollama

The same `--remote-endpoint` path works against any OpenAI-compatible server. To benchmark a vLLM deployment, serve the model and point `gbench` at port 8000:

```bash
# Terminal 1: serve with vLLM (served model id == the HF id you launch)
vllm serve google/gemma-4-26B-A4B-it

# Terminal 2: benchmark the vLLM endpoint (modest-hardware stress config; see the next cell)
python -m gbench --models google/gemma-4-26B-A4B-it \
        --preset quick \
        --remote-endpoint http://127.0.0.1:8000/v1 \
        --tokenizer google/gemma-4-26B-A4B-it \
        --results-dir ./results_101 \
        --stress-reps 1 --stress-min-samples 5
```

### Tuning the stress ramp to your hardware

The stress ramp finds the max arrival rate the server sustains while keeping up **and** meeting a latency SLO (default P99 TTFT ≤ 5000 ms, P99 inter-token ≤ 200 ms), over 3 sweep reps. On a modest laptop/Colab box the slow model can't gather 15 steady completions per rate, so every rate reads `TOO-FEW` and the knee is 0. For those boxes, relax the **sampling** (not the SLO):

| Flag | Controls | Default | Modest hardware |
|---|---|---|---|
| `--stress-min-samples <n>` | steady completions needed per rate point | `15` | `5` (registers a knee faster on a slow box) |
| `--stress-reps <n>` | sweeps per run (more = tighter CI, slower) | `3` | `1` |
| `--no-stress-test` | skip the ramp entirely | (on) | use when you only want serving latency |
| `--stress-threshold <ms>` | P99 **TTFT** SLO = what "sustainable" means | `5000` | keep near default |
| `--stress-tpot-threshold <ms>` | P99 **inter-token** SLO | `200` | keep near default |

**Keep the SLO sane — it *defines* the knee.** Raising `--stress-threshold` far (e.g. to 20000 ms) lets a hopelessly-queued rate "pass", so the reported knee balloons to ~2× the real capacity: it then measures queue depth, not sustainable performance. Relax the *sampling* knobs above for a slow box, not the SLO.

How it behaves on slow hardware: the sweep is **ascending** (probes low→high and stops at the first SLO miss), so it finds a small sustainable rate without ever flooding the server. If even a single request can't meet the TTFT SLO, a preflight **skips** the sweep with a "below stress floor" message — use `--no-stress-test` there (the serving latency numbers are the meaningful measurement).

In [ ]:
# Modest-hardware stress config (see the tuning table above): a single sweep and a
# lower steady-sample floor so a slow box registers a knee quickly. We keep the
# DEFAULT SLO on purpose - over-relaxing --stress-threshold would let a hopelessly
# queued rate "pass" and report a meaningless (inflated) knee.
!gbench --models gemma4-qat:4b \
        --preset quick \
        --remote-endpoint http://localhost:11434/v1 \
        --tokenizer google/gemma-4-E4B-it \
        --results-dir ./results_101 \
        --stress-reps 1 --stress-min-samples 5

## 8. Inspecting benchmark results in Python

Every `gbench` run outputs JSON summaries and sample traces into a timestamped directory under `--results-dir`. We load `summary.json` (which holds every pillar) and render two tables — serving latency and the stress knee. Core columns:
* **`req/s`**: Completed requests per second.
* **`tok/s`**: Generated output tokens per second.
* **`ttft_ms`**: Time to First Token — shown as the **median (P50)**, matching the CLI summary (for single-stream the mean is inflated by the one-time cold-start request, so median is the representative latency).
* **`tpot_ms`**: Time per Output Token (decode), also median (P50).
* **`reliability`**: non-empty completions / total. An *empty* reply is the model returning 0 tokens (e.g. on a nonsense synthetic prompt) — not an error, but the latency/throughput reflect the non-empty requests only.
* **stress table**: `sustainable_req/s` (the knee held under the SLO), `out_tok/s@knee`, P99 TTFT/ITL at the knee, the SLO, and `~concurrent` in-flight requests (Little's Law).

In [ ]:
import json, glob, os
from pathlib import Path
import pandas as pd

# Find the latest result run folder
results_base = Path("./results_101")
run_dirs = sorted([d for d in results_base.iterdir() if d.is_dir()], key=lambda d: d.stat().st_mtime, reverse=True) if results_base.exists() else []

if not run_dirs:
    print("No benchmark result directory found.")
else:
    latest_dir = run_dirs[0]
    summary_path = latest_dir / "summary.json"

    # summary.json's "models" holds EVERY pillar result (serving, stress, ...) in one
    # list, keyed apart by benchmark_type. We split it into two tables below because
    # the pillars have different schemas: serving reports per-request latency
    # (ttft/tpot), while stress reports a single sustainable-rate knee.
    all_records = []
    if summary_path.exists():
        with open(summary_path, "r", encoding="utf-8") as f:
            all_records = json.load(f).get("models", [])

    # ---- Serving: single-stream latency -------------------------------------
    # Keep only serving rows here - a stress/throughput row has no ttft/tpot and
    # would render as a confusing all-zero line. Fall back to raw serve_*.json
    # (no benchmark_type, so they default to serving) when summary.json is absent.
    serving = [r for r in all_records
               if r.get("benchmark_type", "serving") in ("serving", "serving_multimodal")]
    if not serving:
        for pf in (sorted(latest_dir.glob("performance/serve_*.json")) or sorted(latest_dir.glob("serve_*.json"))):
            with open(pf, "r", encoding="utf-8") as f:
                serving.append(json.load(f))

    if serving:
        rows = []
        for r in serving:
            # Reliability = non-empty completions / offered. The POOLED result names
            # these completed_requests/offered_requests (the raw per-iteration
            # completed/total are pooled away), so read those first and fall back to
            # the legacy names / a reconstruction, mirroring the CLI reporter exactly.
            n = int(r.get("completed_requests") or r.get("pooled_sample_count") or r.get("completed") or 0)
            emptyc = int(r.get("empty_requests") or r.get("empty") or 0)
            errc = int(r.get("failed_requests") or 0)
            offered = int(r.get("offered_requests") or 0) or int(r.get("total") or 0) or (n + emptyc + errc)
            rows.append({
                "model": r.get("model", r.get("model_short", r.get("model_name", "unknown"))),
                "format": r.get("format", "N/A"),
                "batch_size": r.get("batch_size", 1),
                "mode": "Multimodal" if r.get("multimodal", False) else "Text",
                "req/s": round(float(r.get("request_throughput", 0.0)), 2),
                "tok/s": round(float(r.get("output_token_throughput", r.get("output_throughput", 0.0))), 2),
                # Median (P50) to MATCH the CLI summary. For single-stream (batch=1)
                # the mean TTFT is inflated by the one-time cold-start request, so
                # median is the representative "typical" latency. Fall back to mean
                # only if an older result file lacks the median field.
                "ttft_ms": round(float(r.get("median_ttft_ms", r.get("mean_ttft_ms", 0.0))), 1),
                "tpot_ms": round(float(r.get("median_tpot_ms", r.get("mean_tpot_ms", 0.0))), 1),
                # non-empty completions / offered. "empty" = model returned 0 tokens
                # (e.g. on a nonsense synthetic prompt) - not an error, but the
                # latency/throughput above reflect the non-empty requests only.
                "reliability": (f"{n}/{offered}" if offered else "-"),
            })
        print("SERVING - single-stream latency (batch=1); ttft_ms/tpot_ms are P50 (median), matching the CLI summary")
        display(pd.DataFrame(rows))
    else:
        print("No serving benchmark records found in:", latest_dir)

    # ---- Stress: max sustainable arrival rate under SLO ---------------------
    # Different schema from serving (a goodput knee, not per-request latency), so
    # it gets its own table. Fall back to raw stress_*.json if summary.json absent.
    stress = [r for r in all_records
              if r.get("benchmark_type") in ("stress_test", "stress_test_multimodal")]
    if not stress:
        for pf in (sorted(latest_dir.glob("performance/stress_*.json")) or sorted(latest_dir.glob("stress_*.json"))):
            with open(pf, "r", encoding="utf-8") as f:
                stress.append(json.load(f))

    if stress:
        rows = []
        for r in stress:
            knee = float(r.get("max_sustainable_qps", 0.0))          # sustainable arrival rate (req/s)
            out_len = float(r.get("workload_output_length", 0) or 0)  # output tokens per request
            rows.append({
                "model": r.get("model", "unknown"),
                "format": r.get("format", "N/A"),
                "mode": "Multimodal" if r.get("multimodal", False) else "Text",
                "sustainable_req/s": round(knee, 3),
                # Aggregate OUTPUT-token throughput at the sustainable load. Each
                # sustained request emits workload_output_length tokens, so
                # knee(req/s) x out_len(tok/req) = tok/s. NB: the JSON's
                # "max_sustainable_throughput" is REQUESTS/s (achieved request rate),
                # NOT tokens - do not display it as a token rate.
                "out_tok/s@knee": round(knee * out_len, 1),
                "P99_TTFT_ms": round(float(r.get("p99_ttft_ms", 0.0))),
                "P99_ITL_ms": round(float(r.get("p99_itl_ms", 0.0)), 1),
                "SLO_TTFT/ITL_ms": f"{int(r.get('ttft_threshold_ms', 0))}/{int(r.get('itl_threshold_ms', 0))}",
                "~concurrent": round(float(r.get("little_law_n_users", 0.0)), 1),
                "reps": r.get("reps", 0),
            })
        print("\nSTRESS - max sustainable arrival rate under SLO (open-loop QPS knee)")
        print("  sustainable_req/s = knee (arrival rate held under SLO);"
              " out_tok/s@knee = knee x output tokens/request (aggregate decode at that load)")
        display(pd.DataFrame(rows))
    else:
        print("\nNo stress benchmark records found in:", latest_dir,
              "- the run may have used --no-stress-test, or the preflight skipped the",
              "sweep with a 'below stress floor' message (see the run log above).")


## 9. Session cleanup and server shutdown

To prevent VRAM fragmentation and orphan background processes, we cleanly shut down the local Ollama serving engine at the end of every notebook session.

In [ ]:
import subprocess, os

print("Stopping Ollama server processes...")
subprocess.run(["pkill", "-f", "ollama"], check=False)

if os.path.exists("Modelfile.qat"):
    os.remove("Modelfile.qat")

print("Session cleanup complete. VRAM and system memory reclaimed.")